# Bunched-Beam Effective Perveance Analysis

$$K_{\rm eff,peak}/K_0 \approx 1 - \bar{\eta}/B_f$$

Primary interpretation limit for any H2/Kr neutralisation result.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RESULTS_DIR    = _ROOT / 'results'
RUNS_DIR       = _ROOT / 'results'
PLOTS_DIR      = _ROOT / 'plots'
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
_DEFAULTS = {
    'beam energy [keV]':       30.0,
    'beam current [mA]':       10.0,
    'bunching factors B_f':    '1, 2, 3, 5, 8',
    'f_RF [MHz]':              72.0,
    'bunch phase width [deg]': 30.0,
}
print_simulation_config(
    notebook_title='Bunched-Beam Effective Perveance',
    defaults=_DEFAULTS, overrides={},
)


## 1. Beam parameters


In [ ]:
import math
from plasma_column.beam import ProtonBeam
from plasma_column.plotting import setup_publication_style, plot_bunched_beam_keff
setup_publication_style()

beam = ProtonBeam(energy_keV=30.0, current_mA=10.0, radius_m=2e-3)
K0   = beam.perveance_K0
B_f_values = [1.0, 2.0, 3.0, 5.0, 8.0]
f_RF_MHz, phase_width_deg = 72.0, 30.0
T_RF = 1.0 / (f_RF_MHz * 1e6)
dt_b = (phase_width_deg / 360.0) * T_RF
dz_b = beam.velocity * dt_b
print(f'K0={K0:.4e}  beta={beam.beta:.6f}  v={beam.velocity:.4e} m/s')
print(f'T_RF={T_RF*1e9:.3f} ns  dt_b={dt_b*1e9:.3f} ns  dz_b={dz_b*1e3:.2f} mm')


## 2. Load eta_avg(t) from first available run


In [ ]:
import warnings
from plasma_column.diagnostics import load_particle_number_diagnostic, compute_particle_number_metrics

_hist = None
for _cd in sorted(RUNS_DIR.iterdir()):
    for _p in [_cd / 'reducedfiles' / 'ParticleNumber_red.txt',
                _cd / 'neutralization_from_particle_number.csv']:
        if _p.exists():
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                _hist = compute_particle_number_metrics(
                    load_particle_number_diagnostic(_p))
            print(f'Using: {_cd.name} ({len(_hist)} steps)')
            break
    if _hist is not None: break

if _hist is None:
    print('No runs found — using synthetic ramp.')
    _t_ns    = np.linspace(0, 400, 300)
    _eta_avg = np.clip(np.linspace(0, 0.75, 300), 0, 1)
else:
    _t_ns    = _hist['time'].values * 1e9
    _eta_avg = _hist['eta_net'].values.clip(0, 1)


In [ ]:
p, _ = plot_bunched_beam_keff(
    _t_ns, _eta_avg, PLOTS_DIR,
    case_name='bunched_beam_analysis',
    bunching_factors=B_f_values,
)
plt.show()
print('Saved:', p.name)


## 3. Final K_eff,peak table


In [ ]:
eta_f = float(_eta_avg[-1])
rows  = [{'B_f': Bf, 'eta_avg_final': eta_f,
           'K_eff_peak_over_K0': max(0.0, 1.0 - eta_f / Bf)}
          for Bf in B_f_values]
display(pd.DataFrame(rows).set_index('B_f').style.format('{:.4f}'))


## 4. K\_eff/K\_0 vs η — parametric curves for multiple B_f

Shows the required average neutralization fraction to reach any K_eff,peak/K0,peak target.
Green shaded region: >50% perveance reduction achieved.

In [ ]:
from plasma_column.plotting import setup_publication_style
setup_publication_style()

eta_arr = np.linspace(0, 1, 500)
Bf_list = [1.0, 2.0, 3.0, 5.0, 8.0, 10.0]
colors = plt.cm.plasma(np.linspace(0.15, 0.9, len(Bf_list)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.axhspan(0.0, 0.5, alpha=0.12, color="green", label="Target: {\rm eff}/K_0 < 0.5$")
ax.axvline(0.7, ls=":", color="gray", lw=1.5, label="Typical $\eta_{\rm avg}=0.7$")
ax.axhline(0.5, ls="--", color="green", lw=1.0, alpha=0.6)

for Bf, col in zip(Bf_list, colors):
    keff = np.clip(1.0 - eta_arr / Bf, 0.0, 1.0)
    ax.plot(eta_arr, keff, color=col, lw=2.0, label=f"={Bf:.0f}$")

ax.set_xlabel(r"Average Neutralization Fraction $ar{\eta}$", fontsize=12)
ax.set_ylabel(r"{m eff,peak}/K_{0,m peak}$", fontsize=12)
ax.set_title(r"Peak-Bunch Effective Perveance vs Average Neutralization", fontsize=13)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=9, ncol=2, loc="upper right")
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "keff_vs_eta_parametric.png", dpi=150, bbox_inches="tight")
plt.savefig(PLOTS_DIR / "keff_vs_eta_parametric.pdf", bbox_inches="tight")
plt.show()
print("Saved keff_vs_eta_parametric.{png,pdf}")

## 5. Beam perveance K\_0 vs current — energy family

Sweeps I_avg from 1–50 mA at four beam energies.
Red star marks the baseline operating point (30 keV, 10 mA).

In [ ]:
from plasma_column.beam import ProtonBeam

energies_keV = [15.0, 20.0, 30.0, 50.0]
currents_mA  = np.linspace(1.0, 50.0, 200)
colors_e = plt.cm.viridis(np.linspace(0.1, 0.9, len(energies_keV)))

fig, ax = plt.subplots(figsize=(8, 5))
for E_keV, col in zip(energies_keV, colors_e):
    K0_vals = [ProtonBeam(energy_keV=E_keV, current_mA=I).perveance_K0
               for I in currents_mA]
    ax.semilogy(currents_mA, K0_vals, color=col, lw=2, label=f"{E_keV:.0f} keV")

# Mark operating point
K0_op = ProtonBeam(energy_keV=30.0, current_mA=10.0).perveance_K0
ax.plot(10.0, K0_op, "r*", ms=14, zorder=5,
        label=f"Op. point (30 keV, 10 mA)
={K0_op:.2e}$")
ax.axhline(1e-3, ls=":", color="gray", lw=1.0, alpha=0.6,
           label=r"High-perveance threshold (=10^{-3}$)")

ax.set_xlabel(r"Average Beam Current {m avg}$ [mA]", fontsize=12)
ax.set_ylabel(r"Generalized Perveance $", fontsize=12)
ax.set_title(r"Generalized Beam Perveance $ vs Beam Current", fontsize=13)
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, which="both", ls="--", alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "K0_vs_current_energy_family.png", dpi=150, bbox_inches="tight")
plt.savefig(PLOTS_DIR / "K0_vs_current_energy_family.pdf", bbox_inches="tight")
plt.show()
print("Saved K0_vs_current_energy_family.{png,pdf}")

## 6. RF bunch parameter sensitivity

**Left**: Bunch spatial length Δz_b vs RF frequency for three phase widths.  
**Right**: Required η_avg to achieve K_eff,peak/K0,peak = 0.5 as a function of B_f.

In [ ]:
from plasma_column.neutralization import bunch_length_m, proton_beta_gamma_speed

beta, gamma, v_beam = proton_beta_gamma_speed(30.0)
freqs_MHz    = np.linspace(10, 200, 300)
phase_widths = [20.0, 30.0, 45.0]
colors_pw    = ["tab:blue", "tab:orange", "tab:green"]

Bf_vals   = np.linspace(1.0, 10.0, 200)
eta_req   = np.clip(0.5 * Bf_vals, 0.0, 1.0)  # eta_min for 50% reduction

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: bunch length vs RF freq
for pw, col in zip(phase_widths, colors_pw):
    dz_mm = [bunch_length_m(v_beam, f * 1e6, pw) * 1e3 for f in freqs_MHz]
    ax1.plot(freqs_MHz, dz_mm, color=col, lw=2, label=f"{pw:.0f}° phase width")
ax1.plot(72.0, bunch_length_m(v_beam, 72e6, 30.0)*1e3, "r*", ms=14, zorder=5,
         label="Operating point (72 MHz, 30°)")
ax1.set_xlabel("RF Frequency {\rm RF}$ [MHz]", fontsize=12)
ax1.set_ylabel(r"Bunch Length $\Delta z_b$ [mm]", fontsize=12)
ax1.set_title(r"Bunch Spatial Length vs RF Frequency", fontsize=13)
ax1.legend(fontsize=9)
ax1.grid(True, ls="--", alpha=0.4)

# Panel B: required eta vs bunching factor
ax2.plot(Bf_vals, eta_req, "tab:purple", lw=2.5,
         label=r"$\eta_{m min} = 0.5\,B_f$ (for 50% reduction)")
ax2.axhline(1.0, ls="--", color="gray", lw=1.2, label="$\eta=1$ (full neutralization)")
ax2.fill_between(Bf_vals, eta_req, 1.0, where=(eta_req<=1.0),
                 alpha=0.15, color="green", label="Achievable region")
ax2.fill_between(Bf_vals, 1.0, eta_req, where=(eta_req>1.0),
                 alpha=0.2, color="red", label="Unachievable (η>1 required)")
ax2.set_xlabel(r"Bunching Factor $", fontsize=12)
ax2.set_ylabel(r"Required $ar{\eta}_{m avg}$ for {m eff,peak}/K_{0,m peak}=0.5$", fontsize=11)
ax2.set_title(r"Required Neutralization vs Bunching Factor", fontsize=13)
ax2.legend(fontsize=9)
ax2.grid(True, ls="--", alpha=0.4)
ax2.set_xlim(1, 10)
ax2.set_ylim(0, 1.5)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "rf_bunch_sensitivity.png", dpi=150, bbox_inches="tight")
plt.savefig(PLOTS_DIR / "rf_bunch_sensitivity.pdf", bbox_inches="tight")
plt.show()
print("Saved rf_bunch_sensitivity.{png,pdf}")